# What the V-cycle carries at every resolution

`net.forward_levels(...)` runs one forward and hands back every intermediate,
keyed by `(outer cycle, grid level)`. Same call for both families:

| container | state | levels come from |
|---|---|---|
| `MGCDLNet` | `z` | `VCycle` |
| `MGLPDSNet` / `MGGroupLPDS` | `(x, z)` | `PDVCycle` |

Recorded per level: the incoming iterate (`x`, `z`), the outgoing one
(`x_out`, `z_out`), what the level below receives (`x_c`, `z_c`), the FAS
corrections (`pi_x`, `pi_z`), and **`coarse_delta_*`** — what the level below
actually handed back, before `alpha` and prolongation.

`coarse_delta_*` is the one to study. It is the *only* thing the coarse grid
contributes. If it is structureless, or dominated by boundary ringing, the
V-cycle is an expensive way to run a flat stack.

In [ ]:
import json, pathlib, sys
import matplotlib.pyplot as plt
import torch

_root = pathlib.Path.cwd()
if not (_root / "models").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))

from models import build_model
from operators import FFT2D, Mask, Sense
from physics.mask import make_acc_mask

torch.manual_seed(0)

# ---- POINT THESE AT A REAL RUN -------------------------------------------
CONFIG = None      # e.g. "trained_nets/mg_recon/knee/mggrouplpds_R8/config.gen.json"
CKPT   = None      # e.g. "trained_nets/mg_recon/knee/mggrouplpds_R8/net.ckpt"
IMG    = (96, 224) # padded HxW; use (304, 704) for real fastMRI knee
COILS  = 6
# --------------------------------------------------------------------------

if CONFIG is None:
    # A small stand-in so the notebook runs before you have a checkpoint. Every
    # panel below is then ARCHITECTURE at initialisation, not anything learned.
    cfg = {"model": {"type": "MGLPDSNet", "params": dict(
               K=[2, [2, 2, 2]], M=16, C=1, P=3, s=2, degrees=1, lam0=1e-3,
               tau0=0.5, theta0=0.0, alpha0=1.0, is_complex=True,
               preproc="kspace", resize_noise=True)},
           "mri": {"R": 8, "acs_lines": 20, "mask_dist": "uniform"},
           "training": {"val_noise_std": 0.005}}
    print("No CONFIG set -- using a small untrained stand-in.")
else:
    cfg = json.load(open(CONFIG))
    print(f"{cfg['model']['type']}  K={cfg['model']['params'].get('K')}")

net = build_model(cfg).eval()
if CKPT:
    from models.level_trace import trace_forward  # noqa: F401  (import check)
    ck = torch.load(CKPT, map_location="cpu", weights_only=False)
    net.load_state_dict(ck["model_state_dict"])
    print(f"loaded {CKPT}")
else:
    print("NO CKPT: panels show the architecture at init, not learned behaviour.")

In [ ]:
H, W = IMG
p = cfg["model"]["params"]
pad = p["s"] * 2 ** (len(p["K"][1]) - 1)
assert H % pad == 0 and W % pad == 0, f"{H}x{W} must be a multiple of pad_stride={pad}"

g = torch.Generator().manual_seed(1)
smaps = torch.randn(1, COILS, H, W, generator=g, dtype=torch.complex64)
smaps = smaps / (smaps.abs().pow(2).sum(1, keepdim=True).sqrt() + 1e-8)
mask = make_acc_mask(shape=(H, W), accel=cfg["mri"]["R"],
                     acs_lines=cfg["mri"]["acs_lines"],
                     mode=cfg["mri"].get("mask_dist", "uniform"))
mask = mask.reshape(1, 1, H, W).to(torch.complex64)
E = Mask(mask) @ FFT2D() @ Sense(smaps)

yy, xx = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W), indexing="ij")
x_true = (((xx**2 + yy**2) < 0.55).float()
          + 0.45 * ((xx**2 + (yy - 0.25)**2) < 0.12).float()
          + 0.30 * ((xx.abs() < 0.5) & (yy.abs() < 0.06)).float())
x_true = (x_true + 0.02 * torch.randn(H, W, generator=g)).reshape(1, 1, H, W)
x_true = x_true.to(torch.complex64)
y = E(x_true)
sigma = cfg.get("training", {}).get("val_noise_std", 0.005)

out, trace = net.forward_levels(y, E=E, sigma=sigma)
print(trace)
print(trace.summary())

## 1. The iterate at each resolution

The primal `x` (or `z` for `MGCDLNet`) entering each level. Every panel is at
its **own** grid size — nothing is resampled, because the point is the
hierarchy.

In [ ]:
def show2d(t, ax, diverging=False, vlim=None, title=None):
    if t is None:
        ax.text(.5, .5, "n/a", ha="center", va="center", color="0.6");
        ax.set_xticks([]); ax.set_yticks([]); return
    im = (t.abs() if t.is_complex() else t)
    im = im.mean(1)[0] if im.shape[1] > 1 else im[0, 0]   # collapse subbands
    im = im.float().cpu()
    if diverging:
        v = vlim or float(im.abs().max()) or 1.0
        ax.imshow(im, cmap="bwr", vmin=-v, vmax=v, interpolation="nearest")
    else:
        v = vlim or float(im.max()) or 1.0
        ax.imshow(im, cmap="gray", vmin=0, vmax=v, interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    if title: ax.set_title(title, fontsize=9)

def grid_plot(keys, outer=-1, diverging=(), share_row=True, figw=3.1):
    lv = trace.levels
    keys = [k for k in keys if any(trace.get(outer, l, k) is not None for l in lv)]
    if not keys:
        print("none of", keys, "were recorded"); return
    fig, ax = plt.subplots(len(keys), len(lv),
                           figsize=(figw*len(lv), 2.9*len(keys)), squeeze=False)
    for i, k in enumerate(keys):
        ts = [trace.get(outer, l, k) for l in lv]
        vl = None
        if share_row:
            fin = [t for t in ts if t is not None]
            if fin:
                mags = [float((t.abs() if t.is_complex() else t).abs().max()) for t in fin]
                vl = max(mags) or 1.0
        for j, (l, t) in enumerate(zip(lv, ts)):
            sh = trace.grid(outer, l, k)
            show2d(t, ax[i][j], diverging=(k in diverging), vlim=vl,
                   title=(f"level {l}   {sh}" if i == 0 else None))
            if j == 0:
                ax[i][j].set_ylabel(k, fontsize=11)
    fig.suptitle(f"outer cycle {trace._norm_outer(outer)}"
                 + ("   (shared window per row)" if share_row else ""), y=1.0)
    fig.tight_layout(); plt.show()

grid_plot(["x", "z"])

## 2. The FAS correction, and what the coarse grid gave back

`pi_*` is the consistency term that makes the coarse level solve the right
problem. `coarse_delta_*` is the coarse level's actual contribution, before
`alpha` scales it and `P` prolongs it.

Both are signed, so they get a diverging map centred on zero — the sign is the
content.

**`coarse_delta_*` lives on the coarse grid**, so its panel under "level 0" has
the shape of level 1. That is deliberate: it is attributed to the level that
*received* it.

In [ ]:
grid_plot(["pi_x", "pi_z", "coarse_delta_x", "coarse_delta_z"],
          diverging=("pi_x", "pi_z", "coarse_delta_x", "coarse_delta_z"))

## 3. Is the coarse grid earning its cost?

The magnitude of `coarse_delta` relative to the iterate it is added to. Small
everywhere means the V-cycle is doing little that the smoothers were not
already doing; concentrated at the border means it is mostly a padding
artifact.

In [ ]:
print(f"{'outer':>6s}{'level':>7s}{'||coarse_delta_x||/||x_c||':>28s}"
      f"{'||coarse_delta_z||/||z_c||':>28s}")
for o in trace.outers:
    for l in trace.levels:
        d = trace.at(o, l)
        row = []
        for v in ("x", "z"):
            cd, c = d.get(f"coarse_delta_{v}"), d.get(f"{v}_c")
            row.append(float(cd.norm() / (c.norm() + 1e-12)) if
                       (cd is not None and c is not None) else float("nan"))
        if any(r == r for r in row):
            print(f"{o:>6d}{l:>7d}{row[0]:>28.4f}{row[1]:>28.4f}")

## 4. Across outer cycles

The same level, as successive V-cycles refine it. The iterate should settle and
the corrections should shrink; corrections that stay large late are a sign the
outer stack has not converged.

In [ ]:
LEVEL = 0
KEY = "x" if trace.get(-1, LEVEL, "x") is not None else "z"
os_ = trace.outers
fig, ax = plt.subplots(2, len(os_), figsize=(3.1*len(os_), 5.8), squeeze=False)
for j, o in enumerate(os_):
    show2d(trace.get(o, LEVEL, KEY), ax[0][j], title=f"outer {o}")
    cd = trace.get(o, LEVEL, "coarse_delta_x")
    if cd is None:                       # `or` on a tensor is ambiguous
        cd = trace.get(o, LEVEL, "coarse_delta_z")
    show2d(cd, ax[1][j], diverging=True)
ax[0][0].set_ylabel(KEY, fontsize=11)
ax[1][0].set_ylabel("coarse_delta", fontsize=11)
fig.suptitle(f"level {LEVEL} across outer cycles", y=1.0)
fig.tight_layout(); plt.show()

## 5. Reading it

- **`coarse_delta` structureless or ~0** — the coarse level is not contributing;
  the V-cycle is paying for depth it is not using.
- **`coarse_delta` concentrated at the border** — grid-transfer boundary
  artifact rather than signal. `operators/resample.py` pads circularly, so a
  bright frame here means the object touches the FOV edge and wraps.
- **`pi` much larger than `coarse_delta`** — the FAS correction is doing most
  of the work and the coarse solve is mostly undoing it, which is what
  inconsistent transfer operators look like.
- **corrections not shrinking across outer cycles** (section 4) — the outer
  stack has not converged; more outer cycles or a larger `alpha` may help.

To point this at a real run, set `CONFIG` and `CKPT` in the first cell and
`IMG = (304, 704)`. Note the input here is a synthetic phantom with random
sensitivity maps: the physics is right, the anatomy is not.